# Exploratory Data Analysis — Credit Risk Analytics

**Date:** July 2026  
**Dataset:** Home Credit Default Risk (Kaggle)  
**Target Variable:** `TARGET` (1 = default, 0 = repaid)  
**Default Rate:** ~8%

---

## Objective

Understand the data, surface risk patterns, and generate actionable business insights. No modeling is performed in this notebook. Every analysis is designed to answer a specific business question about borrower behavior and portfolio risk.

## Analysis Plan

| # | Task | Business Question |
|---|---|---|
| 1 | Inspect Dataset | Is the data trustworthy and fit for purpose? |
| 2 | Check Missing Values | Are missing data patterns themselves informative about risk? |
| 3 | Check Duplicates | Are there suspiciously identical records? |
| 4 | Detect Outliers | Do extreme values represent data errors or genuine high-risk cases? |
| 5 | Explore Distributions | What does a typical the platform borrower look like? |
| 6 | Analyze Target Imbalance | How does the 92/8 split affect our approach? |
| 7 | Analyze Correlations | Which signals move together, and which independently predict risk? |
| 8 | Produce Business Insights | What should the underwriting team do differently starting Monday? |

---
## 1. Inspect Dataset

### Purpose
Load the `application_train` table, verify data types, assess row/column counts, and get an initial sense of data quality.

### Actions
- Load CSV into DataFrame.
- Call `.info()`, `.head()`, `.describe()` on numeric and categorical columns.
- Print shape, column count, data types.
- Inspect unique values in key categorical columns (`NAME_CONTRACT_TYPE`, `CODE_GENDER`, `NAME_EDUCATION_TYPE`, `NAME_FAMILY_STATUS`).

### Business Questions Answered
- *How many applicants are in the training set?* (~300K)
- *Does the data cover the full range of loan products the platform offers?* (Cash vs. Revolving)
- *Are there any obvious data quality red flags?* (Negative ages, impossible incomes)

### Recommended Visualizations

| Chart | Why Useful | Business Question | Expected Insight |
|---|---|---|---|
| **Data type summary table** | Quick scan of numeric vs. categorical features | "Are categoricals encoded usefully?" | Most features are numeric; a handful of object columns need encoding |
| **Descriptive statistics table** | Identifies min/max outliers and scale differences | "Do any values fall outside reasonable human ranges?" | DAYS_BIRTH may show negative values (days before application); DAYS_EMPLOYED may contain absurd outliers |

### Expected Conclusions
- The dataset is large and structured enough for robust analysis.
- Some columns will require type coercion (e.g., objects to categoricals).
- Initial domain checks pass (e.g., loan amounts in reasonable USD ranges).

---
## 2. Check Missing Values

### Purpose
Quantify missingness per column and determine whether missing-data patterns carry predictive signal.

### Actions
- Compute `missing_count` and `missing_%` for every column, sorted descending.
- Visualize missingness with a heatmap and bar chart.
- Compute the default rate for rows *with* vs. *without* a value for key features (especially `EXT_SOURCE_1`, `EXT_SOURCE_2`, `EXT_SOURCE_3`).

### Business Questions Answered
- *Which features are too sparse to use?* (Threshold: >60% missing → drop or flag only)
- *Is missing data random, or does it signal something about the borrower?* (Missing EXT_SOURCE may indicate "thin credit file" = riskier)
- *Should we impute, flag missingness, or both?*

### Recommended Visualizations

| Chart | Why Useful | Business Question | Expected Insight |
|---|---|---|---|
| **Sorted missing-value bar chart** | Instantly see which columns have the worst missing rates | "Which features should be dropped or flagged?" | `EXT_SOURCE_1` may be ~60% missing; `OWN_CAR_AGE` heavily missing |
| **Missing-value heatmap (first 50 rows)** | Visualize whether missingness is systematic per row | "Do some applicants have complete data and others very little?" | Some borrowers have near-complete data; others are sparse — sparsity itself may be a risk signal |
| **Default rate by missingness flag** | Bar chart comparing default rate: missing vs. present for key features | "Does missing EXT_SOURCE predict higher risk?" | Borrowers with missing external scores likely default at a higher rate |

### Expected Conclusions
- 5–10 columns will have >60% missingness and should be flagged for potential removal or binary missing-indicator encoding.
- Missing `EXT_SOURCE` values are likely non-random — they signal borrowers with limited credit history, who are riskier.
- A missing-indicator feature will be created for high-importance sparse columns.

---
## 3. Check Duplicates

### Purpose
Detect exact duplicate rows or near-duplicate applicant records that could bias analysis.

### Actions
- Check `df.duplicated().sum()` for exact row duplicates.
- Check for duplicate `SK_ID_CURR` (the unique applicant identifier) — there should be none.
- If duplicates found, inspect a sample to understand origin (data entry error, reload artifact, etc.).

### Business Questions Answered
- *Are there duplicate loan applications in the training data?* (Could inflate certain borrower profiles)
- *Is the primary key truly unique?*

### Recommended Visualizations

| Chart | Why Useful | Business Question | Expected Insight |
|---|---|---|---|
| **Duplicate count summary table** | Clear Boolean result | "Is deduplication needed?" | Likely zero or very few duplicates in a Kaggle competition dataset |

### Expected Conclusions
- Expect near-zero exact duplicates (Kaggle datasets are clean).
- `SK_ID_CURR` is unique — confirms we can use it as an index for joins.

---
## 4. Detect Outliers

### Purpose
Identify extreme values in numeric features and determine whether they are data errors or genuine (but rare) borrower profiles.

### Actions
- Use IQR method (`Q1 − 1.5×IQR`, `Q3 + 1.5×IQR`) on key numeric columns.
- Focus on business-critical features: `AMT_INCOME_TOTAL`, `AMT_CREDIT`, `AMT_ANNUITY`, `DAYS_BIRTH`, `DAYS_EMPLOYED`.
- For `DAYS_EMPLOYED`: look for the known artifact in Home Credit data where unemployed applicants are coded as a large positive number (~365,000 days ≈ 1000 years).
- Flag outliers, inspect their default rate vs. inlier default rate.

### Business Questions Answered
- *Are reported incomes realistic?* (Someone earning $10M/year on a $5K loan is suspicious)
- *Are there applicants with suspiciously long employment?* (1000+ years = clearly a placeholder for "unemployed")
- *Do outliers default at a different rate than normal-range borrowers?* (If yes, they carry business-relevant signal)

### Recommended Visualizations

| Chart | Why Useful | Business Question | Expected Insight |
|---|---|---|---|
| **Box plots (log scale for income/credit)** | Shows median, IQR, and fences. Log scale prevents extreme values from compressing the visual | "What is the normal range for income and loan amount?" | Most incomes cluster $50K–$150K; outliers above $500K should be capped |
| **DAYS_EMPLOYED histogram (with artifact annotation)** | Reveals the spike at ~365,000 days | "How many unemployed applicants are miscoded?" | Unemployed applicants are encoded as ~1000 years; this must be recoded to NaN |
| **Scatter: AMT_INCOME_TOTAL vs AMT_CREDIT** | Visualize whether extreme-income cases also take extreme loans | "Are outlier incomes paired with proportionate loans?" | Most high-income borrowers take high loans; mismatched pairs may be data errors |
| **Default rate: outliers vs. inliers table** | Compare risk behavior | "Do outlier applicants default more or less often?" | Outliers on income may be lower risk (wealthy); outliers on DAYS_EMPLOYED are higher risk (unemployed) |

### Expected Conclusions
- `DAYS_EMPLOYED` has a known artifact (~1000-year values = unemployed). These should be recoded as NaN or a separate "unemployed" category.
- `AMT_INCOME_TOTAL` has a long right tail. Capping at the 99.9th percentile is reasonable.
- Outliers in `DAYS_BIRTH` (age > 100) should be dropped or flagged.
- Outlier treatment will be deferred to the Feature Engineering stage; here we only document.

---
## 5. Explore Distributions

### Purpose
Understand the shape and range of every important feature. Build a mental model of what a "typical" the platform borrower looks like.

### Actions
- Plot histograms for all numeric features (with density overlay).
- Plot bar charts for all categorical features.
- Group analysis by target: distribution of `AMT_INCOME_TOTAL` for repaid vs. defaulted.

### Business Questions Answered
- *What is the typical the platform borrower profile?* (Age ~35–50, income $50K–$100K, loan ~$100K–$200K)
- *Are there sharp discontinuities in feature distributions that suggest data artifacts?* (E.g., heaping at round numbers)
- *How do loan amounts, incomes, and annuity payments distribute across the portfolio?*

### Recommended Visualizations

| Chart | Why Useful | Business Question | Expected Insight |
|---|---|---|---|
| **Histogram of AMT_INCOME_TOTAL (log scale)** | Income is right-skewed; log scale reveals the bulk of the distribution | "What income range covers 80% of applicants?" | Most applicants earn $30K–$150K |
| **Histogram of AMT_CREDIT** | Loan amount distribution shows product mix | "What loan sizes are most common?" | Loans cluster around round numbers; bimodal pattern possible (small vs. large loans) |
| **Histogram of DAYS_BIRTH (converted to age in years)** | Age distribution of applicants | "What is the core demographic?" | Applicants are 30–55 years old; very few under 25 or over 70 |
| **Bar chart of NAME_CONTRACT_TYPE** | Loan type mix | "Do we have more cash loans or revolving (credit card) loans?" | Cash loans likely dominate |
| **Bar chart of NAME_EDUCATION_TYPE** | Education profile of borrowers | "Are we lending to a well-educated population?" | Secondary education may be the largest group |
| **KDE overlay: income (repaid vs. defaulted)** | Compare distribution shape by target | "Do lower-income borrowers default at higher rates?" | Defaulted borrowers skew toward lower incomes |

### Expected Conclusions
- A clear demographic profile emerges: near-prime borrowers, ages 30–55, secondary/some higher education, incomes $30K–$150K.
- Distributions by target will hint at which features separate good from bad borrowers even before modeling.
- Certain features may show heaping at round numbers (e.g., loan amounts of $10K, $15K, $20K) — this is normal in consumer lending.

---
## 6. Analyze Target Imbalance

### Purpose
Quantify the class imbalance, understand why it exists, and determine the appropriate evaluation strategy.

### Actions
- Compute `TARGET.value_counts(normalize=True)` → ~92% repay, ~8% default.
- Plot a bar chart of class counts.
- Discuss the **business reason** for imbalance: near-prime lenders *intentionally* keep default rates low through screening. An 8% default rate is *normal* and *healthy* in this industry.
- Discuss implications: accuracy is misleading; we need precision, recall, AUC, and especially precision-at-k for collections use cases.

### Business Questions Answered
- *Is this imbalance natural or a sampling artifact?* (Natural — 8% is realistic for near-prime lending)
- *Should we use accuracy as a metric?* (No — a 92% accurate "predict all 0" model is useless)
- *What metric should the business care about?* (Precision at top 5–10% — "of the 50 highest-risk applicants we flag, how many actually default?")

### Recommended Visualizations

| Chart | Why Useful | Business Question | Expected Insight |
|---|---|---|---|
| **Target class counts bar chart** | Clear visual of the 92/8 split | "How imbalanced is this dataset?" | ~8% default rate is visible and interpretable |
| **Accuracy paradox illustration table** | Shows that predicting all 0s gives 92% accuracy but zero business value | "Why can't we just use accuracy?" | A blind model would miss every single default — catastrophic for the bank |

### Expected Conclusions
- The 8% default rate is realistic and will not be artificially balanced via random undersampling (which discards valuable data).
- Strategy: use class weights during training and tune the decision threshold post-training.
- Evaluation will emphasize AUC and precision-at-k, not accuracy.

---
## 7. Analyze Correlations

### Purpose
Identify which features move together (multicollinearity) and which have the strongest individual relationship with default.

### Actions
- Compute Pearson correlation matrix for all numeric columns.
- Sort features by absolute correlation with `TARGET` to identify top predictors.
- Identify pairs with |r| > 0.7 (high collinearity) that may need attention.
- Compute Spearman rank correlation for robustness to outliers.
- For categorical features, compute Cramér's V or ANOVA F-statistic against `TARGET`.

### Business Questions Answered
- *Which single feature correlates most strongly with default?* (Expected: `EXT_SOURCE_2` or `EXT_SOURCE_3`)
- *Are any features so highly correlated that one should be dropped?* (E.g., `AMT_CREDIT` and `AMT_ANNUITY` are naturally correlated — both capture loan size)
- *Do any features have a *counterintuitive* correlation direction?* (E.g., higher income correlating with higher default — would warrant investigation)

### Recommended Visualizations

| Chart | Why Useful | Business Question | Expected Insight |
|---|---|---|---|
| **Correlation heatmap (all numeric features)** | High-level view of feature relationships | "Which feature groups are internally correlated?" | Income, credit, annuity form a correlated cluster; external scores form another |
| **Top-20 features sorted by |correlation| with TARGET** | Identify the strongest univariate predictors | "Which 5 features should underwriting pay attention to?" | EXT_SOURCE_2, EXT_SOURCE_3, DAYS_BIRTH, DAYS_EMPLOYED, AMT_ANNUITY will dominate |
| **Pairplot of top-5 correlated features (colored by TARGET)** | Visualize feature interactions and class separation | "Can we visually separate defaulters from repayers with just 2–3 features?" | Defaulters cluster in low-EXT_SOURCE, low-income, high-DTI space |
| **Cramér's V heatmap for categoricals** | Measure association between categorical features and target | "Which categorical features differentiate risk?" | `NAME_EDUCATION_TYPE` and `NAME_FAMILY_STATUS` will show moderate association |

### Expected Conclusions
- `EXT_SOURCE_2` and `EXT_SOURCE_3` will be the strongest univariate predictors (negative correlation: higher score = lower default rate).
- `DAYS_BIRTH` (age) will show that older applicants default less.
- `DAYS_EMPLOYED` will show a bimodal pattern where very high values (unemployed) strongly correlate with default.
- High collinearity between `AMT_CREDIT` and `AMT_ANNUITY` is expected (annuity = function of credit × rate). One may be dropped or combined into DTI.
- Correlation is not causation, but these findings guide feature engineering.

---
## 8. Produce Business Insights

### Purpose
Synthesize all EDA findings into actionable takeaways for the underwriting team, collections team, and product team.

### Actions
- Summarize the 5 most important findings.
- Translate each finding into a business recommendation.
- Identify potential quick wins (changes that can be made before the model is built).
- Flag any data quality issues that must be fixed in the next stage.

### Business Questions Answered
- *What can the platform do differently starting tomorrow?* (Quick wins)
- *Which borrower segments are most profitable vs. most dangerous?*
- *What feature engineering ideas does the EDA suggest?*

### Recommended Visualizations

| Chart | Why Useful | Business Question | Expected Insight |
|---|---|---|---|
| **Default rate by EXT_SOURCE_2 decile** | Clear monotonic relationship between credit score and risk | "How much does risk drop as the external score rises?" | Bottom decile: ~20% default; top decile: ~2% default — 10× risk differential |
| **Default rate by age group** | Age segmentation | "Should we market differently to younger vs. older borrowers?" | Applicants under 30 default at 2× the rate of applicants over 50 |
| **Default rate by debt-to-income ratio bucket** | DTI is a direct measure of repayment burden | "At what DTI does risk spike?" | DTI > 0.4 shows sharply higher default rates |
| **Default rate by employment length category** | Employment stability signal | "Should we require minimum employment length?" | Applicants with <1 year at current job default at higher rates |
| **Summary insight table** | One-row-per-finding format for executive presentation | "What are the top-5 takeaways?" | See below |

### Expected Findings Summary Table

| Finding | Business Implication | Recommended Action |
|---|---|---|
| EXT_SOURCE_2 is the strongest risk predictor | Borrowers with low external credit scores are ~10× riskier | Use external score as a mandatory underwriting filter before considering loan amount |
| Young borrowers (<30) default at 2× the rate of older borrowers | Age correlates with financial stability | Consider stricter terms (lower loan-to-income) for applicants under 30 |
| Missing external score data signals higher risk | Thin-file borrowers have unmeasured risk | Do not assume missing = average; flag these applicants for manual review |
| Debt-to-income ratio >0.40 triples default risk | Borrowers spending >40% of income on debt are overextended | Hard cap: decline any application with DTI >0.45 regardless of external score |
| Unemployed applicants (DAYS_EMPLOYED artifact) have very high default rates | Jobless borrowers cannot reliably repay | Require proof of income for any applicant flagged as potentially unemployed |

### Expected Conclusions
- The EDA gives us enough signal to define a **simple rule-based underwriting policy** as a baseline to beat with ML.
- Three features (EXT_SOURCE_2, DTI, age) alone provide meaningful risk separation.
- Data cleaning needs are clear: fix DAYS_EMPLOYED encoding, cap income outliers, handle missing EXT_SOURCE.
- The next stage (Feature Engineering) should focus on aggregating bureau and previous-application tables to build richer signals that complement EXT_SOURCE.

---
## Next Steps

The EDA has identified:

1. **Key risk drivers** — external credit scores, age, DTI, employment status
2. **Data quality issues** — DAYS_EMPLOYED artifact, missing values, outliers
3. **Feature engineering opportunities** — aggregate bureau tables, create domain ratios
4. **Evaluation strategy** — AUC and precision-at-k, not accuracy

These findings feed directly into:
- **Data Cleaning notebook** (02): fix DAYS_EMPLOYED, cap outliers, handle missingness
- **Feature Engineering notebook** (03): create DTI, aggregate bureau/previous data, encode missing flags
- **Model Building notebook** (04): train models informed by the feature shortlist above